# Auditoría de equidad de los modelos COMPAS con Aequitas


In [1]:
# %pip install aequitas


## 1. Dependencias

In [2]:
from __future__ import annotations

from datetime import datetime
from html import escape
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

try:
    from aequitas.group import Group
    from aequitas.bias import Bias
    from aequitas.fairness import Fairness
except ImportError as exc:
    raise ImportError(
        "No se encontró Aequitas. Instálalo con: %pip install aequitas"
    ) from exc

warnings.filterwarnings("ignore", category=FutureWarning, module="aequitas")

try:
    AEQUITAS_VERSION = version("aequitas")
except PackageNotFoundError:
    AEQUITAS_VERSION = "desconocida"

print(f"Aequitas {AEQUITAS_VERSION}")


Aequitas 1.1.0


## 2. Configuración

In [3]:
# Ruta creada por los notebooks 1 y 2.
BASE_DIR = Path("data") / "processed"
LEGACY_BASE_DIR = Path(
    "Evaluacion-de-librerias-para-Inteligencia-Artificial-responsable/data/processed"
)
if not BASE_DIR.exists() and LEGACY_BASE_DIR.exists():
    BASE_DIR = LEGACY_BASE_DIR

# None audita todos los CSV generados. Para auditar solo algunos, escribe, por ejemplo:
# MODELOS_A_AUDITAR = {"LR_8_features", "XGB_8_features"}
MODELOS_A_AUDITAR = None

# Atributos protegidos a auditar y grupos de referencia.
ATTR_COLS = ["race", "sex", "age_cat"]
REF_GROUPS = {
    "race": "Caucasian",
    "sex": "Male",
    "age_cat": "25 - 45",
}

# Regla de paridad: tau <= métrica_del_grupo / métrica_de_referencia <= 1/tau.
TAU = 0.80
ALPHA = 0.05

REPORT_DIR = BASE_DIR / "aequitas_report"
REPORT_FILE = REPORT_DIR / "reporte_aequitas.html"
SUMMARY_FILE = REPORT_DIR / "reporte_aequitas_resumen.csv"

print("Directorio de datos:", BASE_DIR.resolve())
print("Directorio del reporte:", REPORT_DIR.resolve())


Directorio de datos: /home/lalo01jm/Documentos/Proyecto Tecnologico/Evaluacion-de-librerias-para-Inteligencia-Artificial-responsable/data/processed
Directorio del reporte: /home/lalo01jm/Documentos/Proyecto Tecnologico/Evaluacion-de-librerias-para-Inteligencia-Artificial-responsable/data/processed/aequitas_report


## 3. Cargar y validar los archivos de auditoría

In [4]:
REQUIRED_COLUMNS = {
    "entity_id", "model_id", "race", "sex", "age_cat", "label_value", "score"
}


def validate_audit_dataframe(audit_df: pd.DataFrame, source: Path) -> pd.DataFrame:
    """Valida el formato de entrada requerido por Aequitas."""
    missing = REQUIRED_COLUMNS.difference(audit_df.columns)
    if missing:
        raise ValueError(f"{source.name}: faltan columnas requeridas: {sorted(missing)}")

    audit_df = audit_df.copy()
    for col in ("score", "label_value"):
        audit_df[col] = pd.to_numeric(audit_df[col], errors="raise").astype(int)
        invalid = sorted(set(audit_df[col].dropna().unique()) - {0, 1})
        if invalid:
            raise ValueError(
                f"{source.name}: {col} debe ser binaria (0/1); se encontraron {invalid}."
            )

    if audit_df[["score", "label_value"]].isna().any().any():
        raise ValueError(f"{source.name}: score y label_value no pueden contener valores nulos.")

    for attr in ATTR_COLS:
        if audit_df[attr].isna().any():
            raise ValueError(f"{source.name}: {attr} contiene valores nulos.")
        audit_df[attr] = audit_df[attr].astype(str)

    for attr, reference in REF_GROUPS.items():
        if reference not in set(audit_df[attr]):
            raise ValueError(
                f"{source.name}: el grupo de referencia {reference!r} no existe en {attr!r}."
            )

    model_ids = audit_df["model_id"].dropna().astype(str).unique()
    if len(model_ids) != 1:
        raise ValueError(
            f"{source.name}: debe contener exactamente un model_id; se encontraron {model_ids.tolist()}."
        )

    return audit_df


audit_paths = sorted(BASE_DIR.glob("aequitas_audit_*.csv"))
if not audit_paths:
    raise FileNotFoundError(
        "No se encontraron archivos aequitas_audit_*.csv. "
        "Ejecuta primero 1.Preprocesamiento_Aequitas.ipynb y "
        "2.Entrenamiento_Modelos_Aequitas.ipynb."
    )

input_audits: dict[str, pd.DataFrame] = {}
for path in audit_paths:
    frame = validate_audit_dataframe(pd.read_csv(path), path)
    model_id = frame["model_id"].iloc[0]
    if MODELOS_A_AUDITAR is None or model_id in MODELOS_A_AUDITAR:
        input_audits[model_id] = frame

if not input_audits:
    raise ValueError("Ningún archivo coincide con MODELOS_A_AUDITAR.")

inventory = pd.DataFrame(
    [
        {
            "model_id": model_id,
            "observaciones": len(frame),
            "positivos_reales": int(frame["label_value"].sum()),
            "positivos_predichos": int(frame["score"].sum()),
            "archivo": next(p.name for p in audit_paths if model_id in pd.read_csv(p, usecols=["model_id"])["model_id"].astype(str).unique()),
        }
        for model_id, frame in input_audits.items()
    ]
).sort_values("model_id").reset_index(drop=True)

display(inventory)


,model_id,observaciones,positivos_reales,positivos_predichos,archivo
0,LR_8_features,6172,2809,2323,aequitas_audit_LR_8_features.csv


## 4. Ejecutar la auditoría

Se usa `Group()` para las métricas absolutas por grupo, `Bias()` para comparar cada grupo contra su referencia y `Fairness()` para evaluar paridades. La referencia para raza es `Caucasian`, para sexo `Male` y para categoría de edad `25 - 45`.

In [6]:
# Prepare input for Aequitas (same columns used elsewhere in the notebook)
aequitas_input = input_audits["LR_8_features"].loc[
    :, ["model_id", "score", "label_value", *ATTR_COLS]
].copy()

group_ = Group()
group_metrics, _ = group_.get_crosstabs(aequitas_input, attr_cols=ATTR_COLS)

bias = Bias()
disparities = bias.get_disparity_predefined_groups(
    group_metrics,
    original_df=aequitas_input,
    ref_groups_dict=REF_GROUPS,
    check_significance=True,
    alpha=ALPHA,
    mask_significance=True,
)

display(group_metrics)
display(disparities)

,model_id,score_threshold,k,attribute_name,attribute_value,accuracy,tpr,tnr,for,fdr,...,pprev,fp,fn,tn,tp,group_label_pos,group_label_neg,group_size,total_entities,prev
0,LR_8_features,binary 0/1,2323,race,African-American,0.680630,0.702589,0.656539,0.331989,0.308239,...,0.531339,520,494,994,1167,1661,1514,3175,6172,0.523150
1,LR_8_features,binary 0/1,2323,race,Asian,0.774194,0.375000,0.913043,0.192308,0.400000,...,0.161290,2,5,21,3,8,23,31,6172,0.258065
2,LR_8_features,binary 0/1,2323,race,Caucasian,0.677603,0.375912,0.871194,0.314917,0.348101,...,0.225392,165,513,1116,309,822,1281,2103,6172,0.390870
3,LR_8_features,binary 0/1,2323,race,Hispanic,0.681729,0.354497,0.875000,0.303483,0.373832,...,0.210216,40,122,280,67,189,320,509,6172,0.371316
4,LR_8_features,binary 0/1,2323,race,Native American,0.727273,0.600000,0.833333,0.285714,0.250000,...,0.363636,1,2,5,3,5,6,11,6172,0.454545
5,LR_8_features,binary 0/1,2323,race,Other,0.685131,0.250000,0.931507,0.313131,0.326087,...,0.134111,15,93,204,31,124,219,343,6172,0.361516
6,LR_8_features,binary 0/1,2323,sex,Female,0.703830,0.276029,0.935696,0.295455,0.300613,...,0.138723,49,299,713,114,413,762,1175,6172,0.351489
7,LR_8_features,binary 0/1,2323,sex,Male,0.675005,0.611853,0.733180,0.327811,0.321296,...,0.432259,694,930,1907,1466,2396,2601,4997,6172,0.479488
8,LR_8_features,binary 0/1,2323,age_cat,25 - 45,0.679502,0.574650,0.770492,0.323898,0.315178,...,0.389864,434,698,1457,943,1641,1891,3532,6172,0.464609
9,LR_8_features,binary 0/1,2323,age_cat,Greater than 45,0.720804,0.275362,0.930603,0.268336,0.348571,...,0.135344,61,300,818,114,414,879,1293,6172,0.320186


,model_id,score_threshold,k,attribute_name,attribute_value,accuracy,tpr,tnr,for,fdr,...,pprev_significance,precision_disparity,precision_ref_group_value,precision_significance,tnr_disparity,tnr_ref_group_value,tnr_significance,tpr_disparity,tpr_ref_group_value,tpr_significance
0,LR_8_features,binary 0/1,2323,race,African-American,0.680630,0.702589,0.656539,0.331989,0.308239,...,True,1.061147,Caucasian,False,0.753608,Caucasian,False,1.869023,Caucasian,False
1,LR_8_features,binary 0/1,2323,race,Asian,0.774194,0.375000,0.913043,0.192308,0.400000,...,False,0.920388,Caucasian,False,1.048036,Caucasian,False,0.997573,Caucasian,False
2,LR_8_features,binary 0/1,2323,race,Caucasian,0.677603,0.375912,0.871194,0.314917,0.348101,...,False,1.000000,Caucasian,False,1.000000,Caucasian,False,1.000000,Caucasian,False
3,LR_8_features,binary 0/1,2323,race,Hispanic,0.681729,0.354497,0.875000,0.303483,0.373832,...,False,0.960530,Caucasian,False,1.004368,Caucasian,False,0.943032,Caucasian,False
4,LR_8_features,binary 0/1,2323,race,Native American,0.727273,0.600000,0.833333,0.285714,0.250000,...,False,1.150485,Caucasian,False,0.956541,Caucasian,False,1.596117,Caucasian,False
5,LR_8_features,binary 0/1,2323,race,Other,0.685131,0.250000,0.931507,0.313131,0.326087,...,True,1.033770,Caucasian,False,1.069230,Caucasian,False,0.665049,Caucasian,False
6,LR_8_features,binary 0/1,2323,sex,Female,0.703830,0.276029,0.935696,0.295455,0.300613,...,True,1.030474,Male,False,1.276216,Male,False,0.451136,Male,False
7,LR_8_features,binary 0/1,2323,sex,Male,0.675005,0.611853,0.733180,0.327811,0.321296,...,False,1.000000,Male,False,1.000000,Male,False,1.000000,Male,False
8,LR_8_features,binary 0/1,2323,age_cat,25 - 45,0.679502,0.574650,0.770492,0.323898,0.315178,...,False,1.000000,25 - 45,False,1.000000,25 - 45,False,1.000000,25 - 45,False
9,LR_8_features,binary 0/1,2323,age_cat,Greater than 45,0.720804,0.275362,0.930603,0.268336,0.348571,...,True,0.951238,25 - 45,False,1.207804,25 - 45,False,0.479183,25 - 45,True


In [7]:
def audit_model(audit_df: pd.DataFrame) -> dict[str, pd.DataFrame | dict[str, object]]:
    """Ejecuta Group, Bias y Fairness sobre un modelo."""
    group = Group()
    bias = Bias()
    fairness = Fairness()

    # Aequitas debe recibir solamente las columnas semánticas de la auditoría.
    # fold y feature_set son metadatos del experimento, no atributos de comparación.
    aequitas_input = audit_df.loc[
        :, ["model_id", "score", "label_value", *ATTR_COLS]
    ].copy()

    group_metrics, _ = group.get_crosstabs(aequitas_input, attr_cols=ATTR_COLS)

    disparities = bias.get_disparity_predefined_groups(
        group_metrics,
        original_df=aequitas_input,
        ref_groups_dict=REF_GROUPS,
        check_significance=True,
        alpha=ALPHA,
        mask_significance=True,
    )

    group_fairness = fairness.get_group_value_fairness(disparities, tau=TAU)
    attribute_fairness = fairness.get_group_attribute_fairness(group_fairness)
    overall_fairness = fairness.get_overall_fairness(attribute_fairness)

    return {
        "group_metrics": group_metrics,
        "disparities": disparities,
        "group_fairness": group_fairness,
        "attribute_fairness": attribute_fairness,
        "overall_fairness": overall_fairness,
    }


audit_results: dict[str, dict[str, pd.DataFrame | dict[str, object]]] = {}
for model_id, audit_df in input_audits.items():
    print(f"Auditando: {model_id}")
    audit_results[model_id] = audit_model(audit_df)

print(f"Modelos auditados: {len(audit_results)}")


Auditando: LR_8_features
Modelos auditados: 1


## 5. Revisar métricas y disparidades

In [8]:
ABSOLUTE_COLUMNS = [
    "model_id", "attribute_name", "attribute_value", "group_size", "prev", "pprev",
    "tpr", "tnr", "fpr", "fnr", "fdr", "for", "precision", "npv",
]
DISPARITY_COLUMNS = [
    "model_id", "attribute_name", "attribute_value", "group_size",
    "fpr_disparity", "fnr_disparity", "pprev_disparity", "fdr_disparity",
    "precision_disparity", "fpr_ref_group_value", "fnr_ref_group_value",
]
FAIRNESS_COLUMNS = [
    "model_id", "attribute_name", "attribute_value", "group_size",
    "FPR Parity", "FNR Parity", "Statistical Parity", "Impact Parity",
    "Equalized Odds", "TypeI Parity", "TypeII Parity",
    "Unsupervised Fairness", "Supervised Fairness",
]

def existing_columns(frame: pd.DataFrame, wanted: list[str]) -> list[str]:
    return [column for column in wanted if column in frame.columns]

for model_id, result in audit_results.items():
    print(f"\n### {model_id}: métricas por grupo")
    display(
        result["group_metrics"]
        .loc[:, existing_columns(result["group_metrics"], ABSOLUTE_COLUMNS)]
        .round(4)
    )

    print(f"\n### {model_id}: disparidades frente a los grupos de referencia")
    display(
        result["disparities"]
        .loc[:, existing_columns(result["disparities"], DISPARITY_COLUMNS)]
        .round(4)
    )

    print(f"\n### {model_id}: paridades por grupo")
    display(
        result["group_fairness"]
        .loc[:, existing_columns(result["group_fairness"], FAIRNESS_COLUMNS)]
    )

    print("Equidad general:", result["overall_fairness"])



### LR_8_features: métricas por grupo


,model_id,attribute_name,attribute_value,group_size,prev,pprev,tpr,tnr,fpr,fnr,fdr,for,precision,npv
0,LR_8_features,race,African-American,3175,0.5231,0.5313,0.7026,0.6565,0.3435,0.2974,0.3082,0.3320,0.6918,0.6680
1,LR_8_features,race,Asian,31,0.2581,0.1613,0.3750,0.9130,0.0870,0.6250,0.4000,0.1923,0.6000,0.8077
2,LR_8_features,race,Caucasian,2103,0.3909,0.2254,0.3759,0.8712,0.1288,0.6241,0.3481,0.3149,0.6519,0.6851
3,LR_8_features,race,Hispanic,509,0.3713,0.2102,0.3545,0.8750,0.1250,0.6455,0.3738,0.3035,0.6262,0.6965
4,LR_8_features,race,Native American,11,0.4545,0.3636,0.6000,0.8333,0.1667,0.4000,0.2500,0.2857,0.7500,0.7143
5,LR_8_features,race,Other,343,0.3615,0.1341,0.2500,0.9315,0.0685,0.7500,0.3261,0.3131,0.6739,0.6869
6,LR_8_features,sex,Female,1175,0.3515,0.1387,0.2760,0.9357,0.0643,0.7240,0.3006,0.2955,0.6994,0.7045
7,LR_8_features,sex,Male,4997,0.4795,0.4323,0.6119,0.7332,0.2668,0.3881,0.3213,0.3278,0.6787,0.6722
8,LR_8_features,age_cat,25 - 45,3532,0.4646,0.3899,0.5746,0.7705,0.2295,0.4254,0.3152,0.3239,0.6848,0.6761
9,LR_8_features,age_cat,Greater than 45,1293,0.3202,0.1353,0.2754,0.9306,0.0694,0.7246,0.3486,0.2683,0.6514,0.7317



### LR_8_features: disparidades frente a los grupos de referencia


,model_id,attribute_name,attribute_value,group_size,fpr_disparity,fnr_disparity,pprev_disparity,fdr_disparity,precision_disparity,fpr_ref_group_value,fnr_ref_group_value
0,LR_8_features,race,African-American,3175,2.6665,0.4766,2.3574,0.8855,1.0611,Caucasian,Caucasian
1,LR_8_features,race,Asian,31,0.6751,1.0015,0.7156,1.1491,0.9204,Caucasian,Caucasian
2,LR_8_features,race,Caucasian,2103,1.0000,1.0000,1.0000,1.0000,1.0000,Caucasian,Caucasian
3,LR_8_features,race,Hispanic,509,0.9705,1.0343,0.9327,1.0739,0.9605,Caucasian,Caucasian
4,LR_8_features,race,Native American,11,1.2939,0.6409,1.6133,0.7182,1.1505,Caucasian,Caucasian
5,LR_8_features,race,Other,343,0.5318,1.2018,0.5950,0.9368,1.0338,Caucasian,Caucasian
6,LR_8_features,sex,Female,1175,0.2410,1.8652,0.3209,0.9356,1.0305,Male,Male
7,LR_8_features,sex,Male,4997,1.0000,1.0000,1.0000,1.0000,1.0000,Male,Male
8,LR_8_features,age_cat,25 - 45,3532,1.0000,1.0000,1.0000,1.0000,1.0000,25 - 45,25 - 45
9,LR_8_features,age_cat,Greater than 45,1293,0.3024,1.7036,0.3472,1.1060,0.9512,25 - 45,25 - 45



### LR_8_features: paridades por grupo


,model_id,attribute_name,attribute_value,group_size,FPR Parity,FNR Parity,Statistical Parity,Impact Parity,Equalized Odds,TypeI Parity,TypeII Parity,Unsupervised Fairness,Supervised Fairness
0,LR_8_features,race,African-American,3175,False,False,False,False,False,False,False,False,False
1,LR_8_features,race,Asian,31,False,True,False,False,False,False,False,False,False
2,LR_8_features,race,Caucasian,2103,True,True,True,True,True,True,True,True,True
3,LR_8_features,race,Hispanic,509,True,True,False,True,True,True,True,False,True
4,LR_8_features,race,Native American,11,False,False,False,False,False,False,False,False,False
5,LR_8_features,race,Other,343,False,True,False,False,False,False,True,False,False
6,LR_8_features,sex,Female,1175,False,False,False,False,False,False,False,False,False
7,LR_8_features,sex,Male,4997,True,True,True,True,True,True,True,True,True
8,LR_8_features,age_cat,25 - 45,3532,True,True,True,True,True,True,True,True,True
9,LR_8_features,age_cat,Greater than 45,1293,False,False,False,False,False,False,False,False,False


Equidad general: {'Unsupervised Fairness': False, 'Supervised Fairness': False, 'Overall Fairness': False}


In [9]:
result

{'group_metrics':          model_id score_threshold     k attribute_name   attribute_value  \
 0   LR_8_features      binary 0/1  2323           race  African-American   
 1   LR_8_features      binary 0/1  2323           race             Asian   
 2   LR_8_features      binary 0/1  2323           race         Caucasian   
 3   LR_8_features      binary 0/1  2323           race          Hispanic   
 4   LR_8_features      binary 0/1  2323           race   Native American   
 5   LR_8_features      binary 0/1  2323           race             Other   
 6   LR_8_features      binary 0/1  2323            sex            Female   
 7   LR_8_features      binary 0/1  2323            sex              Male   
 8   LR_8_features      binary 0/1  2323        age_cat           25 - 45   
 9   LR_8_features      binary 0/1  2323        age_cat   Greater than 45   
 10  LR_8_features      binary 0/1  2323        age_cat      Less than 25   
 
     accuracy       tpr       tnr       for       fdr  ..